## Utilizing Serper API for use with ollama

Since open sourcer LLM's first came out, I ahve always been fascinated by running them on my local PC.  A couple years ago, I purchased an RTX 3090 with 24GB of memory to run relatively larger models with over 20 billion paramters.  Other than the advantages of not having to pay a subsription to an external provider and not sending your data to an external company, it is typically easier to utilize one of the external proviers.   AS seen in the previous example, the tools provided by the OpenAI agents SDK just work when leveraging the OpenAI services, but did not using ollama.

Regardless, my determination to use my local PC's llm continues. This code will utilize a service named Serper API (https://serper.dev/) to perform web seearches.  You can sign up for a free 2500 API calls.  But in this space, it is hard to do everything for free.  A subscription is required once you consumer the free API calls.  

### Increasing the context length of the local model
After running this the first time, I found some strange behavior, like it found flights from Cleveland instead of Columbus and it forgot to save the results to disk.  Increasing the context length seemed to fix the issues. See the md file in this directory to see how to update the context length of the model that ollama uses.

I am impressed the results.  See trip_plan_using_ollama_and_serper.md.


In [1]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from openai import AsyncOpenAI
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from instructions import TripPlannerInstructions

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))



Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


# Use SerperDev as a MCP server

It would be possible to create functions to define the tools avainable to our agents.  But, it will be easier to leverate an MCP server, which will expose all of the capabilities of SerperDev to our agent (https://github.com/garylab/serper-mcp-server). Let's see what tools are available when using the serper MCP server.

The serper.dev web site also has a "Logs" page where you can see the searches that were performed.  It is interesting to see what searches the agent performs.


In [2]:
serper_params = {"command": "uvx", "args": ["serper-mcp-server"], "env": {"SERPER_API_KEY": os.environ.get('SERPER_API_KEY')}}

async with MCPServerStdio(params=serper_params,client_session_timeout_seconds=60) as server:
    serper_tools = await server.list_tools()

serper_tools

[Tool(name='google_search', title=None, description='Search Google for results', inputSchema={'properties': {'q': {'description': 'The query to search for', 'title': 'Q', 'type': 'string'}, 'gl': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The country to search in, e.g. us, uk, ca, au, etc.', 'title': 'Gl'}, 'location': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The location to search in, e.g. San Francisco, CA, USA', 'title': 'Location'}, 'hl': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The language to search in, e.g. en, es, fr, de, etc.', 'title': 'Hl'}, 'page': {'anyOf': [{'pattern': '^[1-9]\\d*$', 'type': 'string'}, {'type': 'null'}], 'default': '1', 'description': 'The page number to return, first page is 1 (integer value as string)', 'title': 'Page'}, 'tbs': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The time period to sea

## Add the Serper MCP server to the agent.



In [4]:

# Generate custom instructions for the trip planner agent
planner = TripPlannerInstructions(
    output_file="trip_plan_using_ollama_and_serper.md"
)
custom_instructions = planner.get_instructions()
print(custom_instructions)


# Set the base_url to your local Ollama instance
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Set a dummy API key (required by the SDK, but not used by Ollama)
DUMMY_API_KEY = "ollama"

# Initialize the AsyncOpenAI client with the custom base_url
client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=DUMMY_API_KEY,
)

# Specify the model you pulled with Ollama
# Note: Ollama expects just the model name (e.g., "llama3"), 
# not the full "gpt-oss" naming convention from the OpenAI API
#OLLAMA_MODEL_NAME = "gpt-oss:20b" 
OLLAMA_MODEL_NAME = "gpt-oss_131k_context:20b" 

# Wrap the client in the Agents SDK model class
model = OpenAIChatCompletionsModel(
    openai_client=client,
    model=OLLAMA_MODEL_NAME
)


sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

#web_search_tool = WebSearchTool(search_context_size="low")

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    async with MCPServerStdio(params=serper_params, client_session_timeout_seconds=45) as mcp_server_serper:
        trip_planner_agent = Agent(
            model=model,
            name="Trip Planner Agent",
            instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
            mcp_servers=[mcp_server_files, mcp_server_serper]
        )
        with trace("Trip Planner Agent Ollama"):
            result = await Runner.run(trip_planner_agent, custom_instructions)
            print(result.final_output)

You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

CRITICAL: You must complete the ENTIRE itinerary before finishing. Do not stop at research phase. Do not ask for permission to continue. Work through all steps until you have a fully detailed day-by-day schedule.

The customer has provided the following details for their trip:
- Home Location: Columbus, OH
- Departure Date: 2026/02/25
- Return Date: 2026/03/07
- Destination: Tokyo, Japan
- Must-Do Activities: Visit the Tokyo Tower, Explore Akihabara, Experience a traditional tea ceremony, Visit the Tsukiji Fish Market, Take a day trip to Mount Fuji   
- Number of Travelers: 3
- Ages of Travelers: 51, 50, 17
- Other Considerations: 
  - I will be running in the Tokyo marathon on Sunday March 1, so I only need a relaxing place to eat on that day.
  - Starting on March 3, throughout the r